# Pearls AQI Predictor — Exploratory Data Analysis
Trends, seasonality, autocorrelation and pollutant relationships used to design features.

Runs offline on the synthetic generator; with API keys it reads the live feature store.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.feature_pipeline.data_fetcher import DataFetcher
from src.feature_pipeline.feature_engineer import engineer_features

df = DataFetcher().fetch_history('london', days=90)
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(df.shape); df.head()

## 1. AQI over time + distribution

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(df['timestamp'], df['aqi']); ax[0].set_title('AQI over time'); ax[0].set_ylabel('AQI')
ax[1].hist(df['aqi'], bins=40); ax[1].set_title('AQI distribution')
plt.tight_layout(); plt.show()
df['aqi'].describe()

## 2. Daily + weekly seasonality

In [ ]:
df['hour'] = df['timestamp'].dt.hour
df['dow']  = df['timestamp'].dt.dayofweek
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
df.groupby('hour')['aqi'].mean().plot(ax=ax[0], marker='o', title='Mean AQI by hour')
df.groupby('dow')['aqi'].mean().plot(ax=ax[1], marker='o', title='Mean AQI by day of week')
plt.tight_layout(); plt.show()

## 3. Autocorrelation — motivates lag/rolling features

In [ ]:
lags = range(1, 49)
acf = [df['aqi'].autocorr(lag=l) for l in lags]
plt.figure(figsize=(12, 4)); plt.bar(list(lags), acf)
plt.title('AQI autocorrelation (lags 1-48h)'); plt.xlabel('lag (h)'); plt.show()

## 4. Pollutant + weather correlations

In [ ]:
cols = ['aqi','pm25','pm10','o3','no2','so2','co','temp','humidity','wind_speed']
corr = df[cols].corr()
plt.figure(figsize=(8, 6)); plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.xticks(range(len(cols)), cols, rotation=90); plt.yticks(range(len(cols)), cols)
plt.colorbar(); plt.title('Correlation matrix'); plt.tight_layout(); plt.show()
corr['aqi'].sort_values(ascending=False)

## 5. Engineered feature preview

In [ ]:
feat = engineer_features(df)
print('engineered columns:', len(feat.columns))
feat[['timestamp','aqi','aqi_lag_1','aqi_roll_mean_24','aqi_change_rate','wind_dispersion']].tail()

### Takeaways
- Strong daily (afternoon peak) + weekly cycles → cyclical encodings.
- High short-lag autocorrelation → lag_{1..24} + rolling stats are the dominant predictors.
- pm2.5 is the primary AQI driver → confirms EPA-breakpoint backfill.
- Per-city scale differences → train one model per city.